In [19]:
import os
from pathlib import Path

from dotenv import load_dotenv

workspace_root = Path.cwd().parent
env_paths = (
    Path.cwd() / ".env",
    workspace_root / ".env",
    workspace_root / "Langchain_Basics" / ".env",
)

for env_path in env_paths:
    if env_path.exists():
        load_dotenv(env_path, override=True)
        print(f"Loaded environment from: {env_path}")
        break
else:
    print("No .env file found. Create Agents/.env or workspace-root/.env.")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_PROJECT", "LangChainTrainings-Agents")

if os.getenv("LANGSMITH_API_KEY"):
    print(f"LangSmith tracing enabled for project: {os.environ['LANGSMITH_PROJECT']}")
else:
    print("Add LANGSMITH_API_KEY to .env to enable LangSmith tracing.")

Loaded environment from: c:\Users\Girish Kulkarni\Downloads\LangChainTrainings\Langchain_Basics\.env
LangSmith tracing enabled for project: Firstproject


In [20]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    base_url="http://localhost:11434",
    model="qwen3:8b",
    temperature=0.5,
    num_predict=2500,
    reasoning=False,
)

In [21]:
# Wikipedia tool with retry handling

import json
import time

from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import WikipediaQueryRun

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=4000,
    )
)


def wikipedia_invoke_with_retry(query, max_attempts=3):
    for attempt in range(max_attempts):
        try:
            return wikipedia.invoke(query)
        except (json.JSONDecodeError, ConnectionError, TimeoutError) as error:
            if attempt == max_attempts - 1:
                return f"Wikipedia request failed after {max_attempts} attempts: {error}"
            time.sleep(2 ** attempt)


tool_response = wikipedia_invoke_with_retry("What is the capital of India?")
tool_response

'Wikipedia request failed after 3 attempts: Expecting value: line 1 column 1 (char 0)'

In [22]:
# Simple DuckDuckGo search

from langchain_community.tools import DuckDuckGoSearchRun

duckduckgo_search = DuckDuckGoSearchRun()

search_result = duckduckgo_search.invoke("Where is the Eiffel Tower located?")
print(search_result)

1.4Inauguration and the 1889 exposition. 1.5Subsequent events. 2Design. Toggle Design subsection. 2.1Material. Oct 22, 2025 · Location: The Eiffel Tower is situated on the Champ de Mars, a large public green space in the 7th arrondissement of Paris, near the banks of the Seine River. It stands prominently in the Parisian skyline and is visible from various parts of the city. The Editors of Encyclopaedia Britannica The Eiffel Tower can be found on the Champs de Mars at 5 Avenue Anatole France within the 7th arrondissement of Paris. Aug 8, 2026 · Eiffel Tower, wrought-iron structure in Paris that is one of the most famous landmarks in the world. It is also a technological masterpiece in building-construction history. Nov 13, 2025 · Find the Eiffel Tower’s exact location in Paris—address, GPS coordinates, nearest Metro/RER, and the best approaches for views—plus UNESCO context and tips. The Eiffel Tower is located in the seventh Arrondissement of Paris, an elegant district with wide avenu

In [23]:
# Creatin custome tools

from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def substarct(a: int, b: int) -> int:
    """Add two numbers."""
    return a - b

@tool
def multiply(a: int, b: int) -> int:
    """Add two numbers."""
    return a * b

print(add.invoke({"a":10, "b": 20}))  # Example usage of the custom tool



30


In [24]:
tools = [wikipedia, add, substarct, multiply]

list_of_tools = {tool.name: tool for tool in tools}

llm_with_tools = llm.bind_tools(tools)

response = llm_with_tools.invoke(
    "What is the capital of India?"
)

response

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-09-08T16:49:01.6590528Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3202518000, 'load_duration': 91742100, 'prompt_eval_count': 311, 'prompt_eval_duration': 696927000, 'eval_count': 22, 'eval_duration': 2398591000, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--01a081ec-365d-7632-94c6-ca804f52cb20-0', tool_calls=[{'name': 'wikipedia', 'args': {'query': 'capital of India'}, 'id': '9cc10f13-d379-4217-b8e2-bad40642414f', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 311, 'output_tokens': 22, 'total_tokens': 333})

In [27]:
# Execute the custom tools with LLM

from langchain_core.messages import SystemMessage, HumanMessage


query = "what is the capital of India? AND 2=4 = ?"

message = [ human_message := HumanMessage(content=query) ]

ai_message = llm_with_tools.invoke(query)

ai_message.tool_calls




[{'name': 'wikipedia',
  'args': {'query': 'capital of India'},
  'id': 'd634309d-960e-457b-9f4d-33d6ca2f1ca3',
  'type': 'tool_call'},
 {'name': 'substarct',
  'args': {'a': 2, 'b': 4},
  'id': '02f935b1-ecdc-42a9-a930-e27eb4ae45ff',
  'type': 'tool_call'}]

In [26]:
# Execute the tools requested by the LLM

for tool_call in ai_message.tool_calls:
    tool_name = tool_call["name"].lower()
    tool_args = tool_call["args"]
    execute_tool = list_of_tools[tool_name]

    if tool_name == "wikipedia":
        tool_result = wikipedia_invoke_with_retry(**tool_args)
    else:
        tool_result = execute_tool.invoke(tool_args)

    print(f"{tool_name}: {tool_result}")

wikipedia: Page: National Capital Region (India)
Summary: The National Capital Region (NCR; Rāṣṭrīya Rājadhānī Kṣetra) is a region centred on the city of Delhi, a special union territory of India that hosts the country's capital city New Delhi. It encompasses the entirety of Delhi and a number of adjacent districts from the states of Haryana, Uttar Pradesh, and Rajasthan. The NCR and the associated National Capital Region Planning Board (NCRPB) were created in 1985 to plan the development of the region and to evolve "harmonized policies for the control of land-uses and development of infrastructure" in the region. Prominent cities of the NCR are Delhi, New Delhi, Faridabad, Gurgaon, Noida, Ghaziabad and Meerut.
The NCR is a mixed, rural-urban region, with a population of over 46,069,000 and an urbanisation of 62.6 percent. There are also areas like the Aravalli ridge, forests, wildlife and bird sanctuaries. The Delhi Extended Urban Agglomeration, the inner part of the NCR, had an estim